In [2]:
# Cell 2
import io
import time
import zipfile
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from pathlib import Path

API_KEY = "7898aa8e16939920341d09eaa93ff9815789bafb"
SAVE_DIR = Path("./dart_output")
SAVE_DIR.mkdir(exist_ok=True)

REPORT_CODES = {
    "사업보고서": "11011",
    "반기보고서": "11012",
    "1분기보고서": "11013",
    "3분기보고서": "11014",
}

BASE_URL = "https://opendart.fss.or.kr/api"

ENDPOINTS = {
    "corp_code": f"{BASE_URL}/corpCode.xml",
    "treasury_stock": f"{BASE_URL}/tesstkAcqsDspsSttus.json",   # 자기주식 취득 및 처분 현황
    "major_shareholder": f"{BASE_URL}/hyslrSttus.json",          # 최대주주 현황
    "major_shareholder_change": f"{BASE_URL}/hyslrChgSttus.json" # 최대주주 변동현황
}

In [3]:
# Cell 3
def safe_request(url, params=None, timeout=30, max_retries=3, sleep_sec=0.2):
    last_err = None
    for i in range(max_retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            time.sleep(sleep_sec)
            return r
        except Exception as e:
            last_err = e
            time.sleep((i + 1) * 1.0)
    raise last_err

def dart_json(url, params):
    r = safe_request(url, params=params)
    data = r.json()
    status = data.get("status")
    message = data.get("message", "")
    if status not in ("000", "013"):
        raise RuntimeError(f"DART 오류: status={status}, message={message}, url={url}, params={params}")
    return data

def list_to_df(data):
    rows = data.get("list", [])
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows)

In [4]:
# Cell 4
def download_corp_codes(api_key: str) -> pd.DataFrame:
    params = {"crtfc_key": api_key}
    r = safe_request(ENDPOINTS["corp_code"], params=params, timeout=60)

    with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
        xml_name = zf.namelist()[0]
        xml_bytes = zf.read(xml_name)

    root = ET.fromstring(xml_bytes)
    rows = []
    for item in root.findall("list"):
        rows.append({
            "corp_code": item.findtext("corp_code", default="").strip(),
            "corp_name": item.findtext("corp_name", default="").strip(),
            "stock_code": item.findtext("stock_code", default="").strip(),
            "modify_date": item.findtext("modify_date", default="").strip(),
        })

    df = pd.DataFrame(rows)
    df["stock_code"] = df["stock_code"].astype(str).str.zfill(6)
    df = df[df["stock_code"] != ""].copy()
    return df

corp_map = download_corp_codes(API_KEY)
corp_map.head()

,corp_code,corp_name,stock_code,modify_date
0,00434003,다코,000000,20170630
1,00430964,굿앤엘에스,000000,20170630
2,00388953,크레디피아제이십오차유동화전문회사,000000,20170630
3,00179984,연방건설산업,000000,20170630
4,00420143,브룩스피알아이오토메이션잉크,000000,20170630


In [5]:
# Cell 5
def get_corp_code_from_stock_code(stock_code: str, corp_map_df: pd.DataFrame) -> str:
    stock_code = str(stock_code).zfill(6)
    hit = corp_map_df.loc[corp_map_df["stock_code"] == stock_code, "corp_code"]
    if hit.empty:
        raise ValueError(f"corp_code를 찾을 수 없습니다: stock_code={stock_code}")
    return hit.iloc[0]

def get_treasury_stock_status(corp_code: str, bsns_year: int, reprt_code: str) -> pd.DataFrame:
    params = {
        "crtfc_key": API_KEY,
        "corp_code": corp_code,
        "bsns_year": str(bsns_year),
        "reprt_code": reprt_code,
    }
    data = dart_json(ENDPOINTS["treasury_stock"], params)
    df = list_to_df(data)
    if not df.empty:
        df["corp_code"] = corp_code
        df["bsns_year"] = bsns_year
        df["reprt_code"] = reprt_code
    return df

def get_major_shareholder_status(corp_code: str, bsns_year: int, reprt_code: str) -> pd.DataFrame:
    params = {
        "crtfc_key": API_KEY,
        "corp_code": corp_code,
        "bsns_year": str(bsns_year),
        "reprt_code": reprt_code,
    }
    data = dart_json(ENDPOINTS["major_shareholder"], params)
    df = list_to_df(data)
    if not df.empty:
        df["corp_code"] = corp_code
        df["bsns_year"] = bsns_year
        df["reprt_code"] = reprt_code
    return df

def get_major_shareholder_change_status(corp_code: str, bsns_year: int, reprt_code: str) -> pd.DataFrame:
    params = {
        "crtfc_key": API_KEY,
        "corp_code": corp_code,
        "bsns_year": str(bsns_year),
        "reprt_code": reprt_code,
    }
    data = dart_json(ENDPOINTS["major_shareholder_change"], params)
    df = list_to_df(data)
    if not df.empty:
        df["corp_code"] = corp_code
        df["bsns_year"] = bsns_year
        df["reprt_code"] = reprt_code
    return df

In [6]:
# Cell 6
def collect_for_one_stock(stock_code: str, years, report_names=None, corp_map_df=None):
    if corp_map_df is None:
        corp_map_df = corp_map

    if report_names is None:
        report_names = ["사업보고서", "반기보고서", "1분기보고서", "3분기보고서"]

    corp_code = get_corp_code_from_stock_code(stock_code, corp_map_df)

    treasury_list = []
    major_list = []
    major_chg_list = []

    for year in years:
        for report_name in report_names:
            reprt_code = REPORT_CODES[report_name]

            try:
                df1 = get_treasury_stock_status(corp_code, year, reprt_code)
                if not df1.empty:
                    df1["stock_code"] = str(stock_code).zfill(6)
                    df1["report_name"] = report_name
                    treasury_list.append(df1)
            except Exception as e:
                print(f"[자사주 오류] {stock_code} {year} {report_name}: {e}")

            try:
                df2 = get_major_shareholder_status(corp_code, year, reprt_code)
                if not df2.empty:
                    df2["stock_code"] = str(stock_code).zfill(6)
                    df2["report_name"] = report_name
                    major_list.append(df2)
            except Exception as e:
                print(f"[최대주주 현황 오류] {stock_code} {year} {report_name}: {e}")

            try:
                df3 = get_major_shareholder_change_status(corp_code, year, reprt_code)
                if not df3.empty:
                    df3["stock_code"] = str(stock_code).zfill(6)
                    df3["report_name"] = report_name
                    major_chg_list.append(df3)
            except Exception as e:
                print(f"[최대주주 변동 오류] {stock_code} {year} {report_name}: {e}")

    treasury_df = pd.concat(treasury_list, ignore_index=True) if treasury_list else pd.DataFrame()
    major_df = pd.concat(major_list, ignore_index=True) if major_list else pd.DataFrame()
    major_chg_df = pd.concat(major_chg_list, ignore_index=True) if major_chg_list else pd.DataFrame()

    return treasury_df, major_df, major_chg_df

In [7]:
# Cell 7
# 예시: 삼성전자(005930), SK하이닉스(000660)
stock_codes = ["005930", "000660"]
years = range(2019, 2026)

all_treasury = []
all_major = []
all_major_chg = []

for sc in stock_codes:
    tdf, mdf, cdf = collect_for_one_stock(sc, years)
    if not tdf.empty:
        all_treasury.append(tdf)
    if not mdf.empty:
        all_major.append(mdf)
    if not cdf.empty:
        all_major_chg.append(cdf)

treasury_df = pd.concat(all_treasury, ignore_index=True) if all_treasury else pd.DataFrame()
major_df = pd.concat(all_major, ignore_index=True) if all_major else pd.DataFrame()
major_chg_df = pd.concat(all_major_chg, ignore_index=True) if all_major_chg else pd.DataFrame()

print("자사주:", treasury_df.shape)
print("최대주주 현황:", major_df.shape)
print("최대주주 변동:", major_chg_df.shape)

자사주: (447, 19)
최대주주 현황: (978, 17)
최대주주 변동: (428, 15)


In [8]:
# Cell 8
treasury_df.head()

,rcept_no,corp_cls,corp_code,corp_name,stock_knd,acqs_mth1,acqs_mth2,acqs_mth3,bsis_qy,change_qy_acqs,change_qy_dsps,change_qy_incnr,trmend_qy,rm,stlm_dt,bsns_year,reprt_code,stock_code,report_name
0,20200330003851,Y,00126380,삼성전자,-,-,-,-,-,-,-,-,-,-,2019-12-31,2019,11011,005930,사업보고서
1,20190814002218,Y,00126380,삼성전자,-,-,-,-,-,-,-,-,-,-,2019-06-30,2019,11012,005930,반기보고서
2,20190515001605,Y,00126380,삼성전자,-,-,-,-,-,-,-,-,-,-,2019-03-31,2019,11013,005930,1분기보고서
3,20191114001273,Y,00126380,삼성전자,-,-,-,-,-,-,-,-,-,-,2019-09-30,2019,11014,005930,3분기보고서
4,20210309000744,Y,00126380,삼성전자,-,-,-,-,-,-,-,-,-,-,2020-12-31,2020,11011,005930,사업보고서


In [9]:
# Cell 9
major_df.head()

,rcept_no,corp_cls,corp_code,corp_name,stock_knd,nm,relate,bsis_posesn_stock_co,bsis_posesn_stock_qota_rt,trmend_posesn_stock_co,trmend_posesn_stock_qota_rt,rm,stlm_dt,bsns_year,reprt_code,stock_code,report_name
0,20200330003851,Y,00126380,삼성전자,보통주,이건희,최대주주 본인,"249,273,200",4.18,"249,273,200",4.18,-,2019-12-31,2019,11011,005930,사업보고서
1,20200330003851,Y,00126380,삼성전자,보통주,삼성생명보험㈜(특별계정),계열회사,"19,048,733",0.32,"18,286,593",0.31,장내매매,2019-12-31,2019,11011,005930,사업보고서
2,20200330003851,Y,00126380,삼성전자,우선주,삼성생명보험㈜(특별계정),계열회사,"1,268,546",0.15,"1,352,563",0.16,장내매매,2019-12-31,2019,11011,005930,사업보고서
3,20200330003851,Y,00126380,삼성전자,보통주,삼성화재해상보험㈜,계열회사,"88,802,052",1.49,"88,802,052",1.49,-,2019-12-31,2019,11011,005930,사업보고서
4,20200330003851,Y,00126380,삼성전자,보통주,이상훈,발행회사 임원,"28,500",0.00,"16,000",0.00,장내매매,2019-12-31,2019,11011,005930,사업보고서


In [10]:
# Cell 10
major_chg_df.head()

,rcept_no,corp_cls,corp_code,corp_name,change_on,mxmm_shrholdr_nm,posesn_stock_co,qota_rt,change_cause,rm,stlm_dt,bsns_year,reprt_code,stock_code,report_name
0,20200330003851,Y,00126380,삼성전자,-,-,-,-,-,-,2019-12-31,2019,11011,005930,사업보고서
1,20190814002218,Y,00126380,삼성전자,-,-,-,-,-,-,2019-06-30,2019,11012,005930,반기보고서
2,20190515001605,Y,00126380,삼성전자,-,-,-,-,-,-,2019-03-31,2019,11013,005930,1분기보고서
3,20191114001273,Y,00126380,삼성전자,-,-,-,-,-,-,2019-09-30,2019,11014,005930,3분기보고서
4,20210309000744,Y,00126380,삼성전자,-,-,-,-,-,-,2020-12-31,2020,11011,005930,사업보고서


In [11]:
# Cell 11
def save_outputs(treasury_df, major_df, major_chg_df, save_dir=SAVE_DIR):
    if not treasury_df.empty:
        treasury_df.to_csv(save_dir / "treasury_stock_status.csv", index=False, encoding="utf-8-sig")
    if not major_df.empty:
        major_df.to_csv(save_dir / "major_shareholder_status.csv", index=False, encoding="utf-8-sig")
    if not major_chg_df.empty:
        major_chg_df.to_csv(save_dir / "major_shareholder_change_status.csv", index=False, encoding="utf-8-sig")

    with pd.ExcelWriter(save_dir / "dart_major_shareholder_treasury.xlsx", engine="openpyxl") as writer:
        (treasury_df if not treasury_df.empty else pd.DataFrame()).to_excel(writer, sheet_name="treasury_stock", index=False)
        (major_df if not major_df.empty else pd.DataFrame()).to_excel(writer, sheet_name="major_shareholder", index=False)
        (major_chg_df if not major_chg_df.empty else pd.DataFrame()).to_excel(writer, sheet_name="major_shareholder_change", index=False)

save_outputs(treasury_df, major_df, major_chg_df)
print("저장 완료:", SAVE_DIR.resolve())

저장 완료: C:\Users\Admin\hipython\project2\dart_output


In [12]:
# Cell 12
# 여러 종목을 한 번에 돌리고 싶을 때 사용하는 함수
def collect_many_stocks(stock_codes, years, report_names=None, corp_map_df=None):
    if corp_map_df is None:
        corp_map_df = corp_map

    treasury_all, major_all, major_chg_all = [], [], []

    for i, stock_code in enumerate(stock_codes, 1):
        print(f"[{i}/{len(stock_codes)}] collecting {stock_code}")
        try:
            tdf, mdf, cdf = collect_for_one_stock(
                stock_code=stock_code,
                years=years,
                report_names=report_names,
                corp_map_df=corp_map_df
            )
            if not tdf.empty:
                treasury_all.append(tdf)
            if not mdf.empty:
                major_all.append(mdf)
            if not cdf.empty:
                major_chg_all.append(cdf)
        except Exception as e:
            print(f"[종목 실패] {stock_code}: {e}")

    treasury_df = pd.concat(treasury_all, ignore_index=True) if treasury_all else pd.DataFrame()
    major_df = pd.concat(major_all, ignore_index=True) if major_all else pd.DataFrame()
    major_chg_df = pd.concat(major_chg_all, ignore_index=True) if major_chg_all else pd.DataFrame()

    return treasury_df, major_df, major_chg_df

In [13]:
# Cell 13
# 네 데이터프레임에 ticker 컬럼이 있다면 이렇게 사용
# sample_stock_codes = sorted(df["ticker"].astype(str).str.zfill(6).unique().tolist())

# 예시 실행
sample_stock_codes = ["005930", "000660", "373220"]
years = range(2019, 2026)

treasury_df2, major_df2, major_chg_df2 = collect_many_stocks(sample_stock_codes, years)

print(treasury_df2.shape, major_df2.shape, major_chg_df2.shape)

[1/3] collecting 005930
[2/3] collecting 000660
[3/3] collecting 373220
(468, 19) (1038, 17) (465, 15)


In [14]:
# Cell 14
# 필요한 컬럼만 간단히 정리하는 예시
def clean_numeric_text(s):
    if pd.isna(s):
        return s
    return str(s).replace(",", "").replace("%", "").strip()

for df_ in [treasury_df, major_df, major_chg_df]:
    if not df_.empty:
        for col in df_.columns:
            if df_[col].dtype == object:
                df_[col] = df_[col].map(clean_numeric_text)

treasury_df.head(3), major_df.head(3), major_chg_df.head(3)

(         rcept_no corp_cls corp_code corp_name stock_knd acqs_mth1 acqs_mth2  \
 0  20200330003851        Y  00126380      삼성전자         -         -         -   
 1  20190814002218        Y  00126380      삼성전자         -         -         -   
 2  20190515001605        Y  00126380      삼성전자         -         -         -   
 
   acqs_mth3 bsis_qy change_qy_acqs change_qy_dsps change_qy_incnr trmend_qy  \
 0         -       -              -              -               -         -   
 1         -       -              -              -               -         -   
 2         -       -              -              -               -         -   
 
   rm     stlm_dt  bsns_year reprt_code stock_code report_name  
 0  -  2019-12-31       2019      11011     005930       사업보고서  
 1  -  2019-06-30       2019      11012     005930       반기보고서  
 2  -  2019-03-31       2019      11013     005930      1분기보고서  ,
          rcept_no corp_cls corp_code corp_name stock_knd             nm  \
 0  20200330003

In [19]:
print("자사주 데이터 건수:", len(treasury_df))
print("최대주주 현황 건수:", len(major_df))
print("최대주주 변동 건수:", len(major_chg_df))

자사주 데이터 건수: 447
최대주주 현황 건수: 978
최대주주 변동 건수: 428


In [20]:
print("자사주 종목 수:", treasury_df["stock_code"].nunique() if not treasury_df.empty else 0)
print("최대주주 현황 종목 수:", major_df["stock_code"].nunique() if not major_df.empty else 0)
print("최대주주 변동 종목 수:", major_chg_df["stock_code"].nunique() if not major_chg_df.empty else 0)

자사주 종목 수: 2
최대주주 현황 종목 수: 2
최대주주 변동 종목 수: 2


In [23]:
print("=== 저장 경로 ===")
print(SAVE_DIR.resolve())

print("\n=== 데이터 크기 ===")
print("treasury_df:", treasury_df.shape)
print("major_df:", major_df.shape)
print("major_chg_df:", major_chg_df.shape)

print("\n=== 컬럼명 ===")
print("treasury_df columns:", treasury_df.columns.tolist())
print("major_df columns:", major_df.columns.tolist())
print("major_chg_df columns:", major_chg_df.columns.tolist())

print("\n=== 미리보기 ===")
display(treasury_df.head(10))
display(major_df.head(10))
display(major_chg_df.head(10))

=== 저장 경로 ===
C:\Users\Admin\hipython\project2\dart_output

=== 데이터 크기 ===
treasury_df: (447, 19)
major_df: (978, 17)
major_chg_df: (428, 15)

=== 컬럼명 ===
treasury_df columns: ['rcept_no', 'corp_cls', 'corp_code', 'corp_name', 'stock_knd', 'acqs_mth1', 'acqs_mth2', 'acqs_mth3', 'bsis_qy', 'change_qy_acqs', 'change_qy_dsps', 'change_qy_incnr', 'trmend_qy', 'rm', 'stlm_dt', 'bsns_year', 'reprt_code', 'stock_code', 'report_name']
major_df columns: ['rcept_no', 'corp_cls', 'corp_code', 'corp_name', 'stock_knd', 'nm', 'relate', 'bsis_posesn_stock_co', 'bsis_posesn_stock_qota_rt', 'trmend_posesn_stock_co', 'trmend_posesn_stock_qota_rt', 'rm', 'stlm_dt', 'bsns_year', 'reprt_code', 'stock_code', 'report_name']
major_chg_df columns: ['rcept_no', 'corp_cls', 'corp_code', 'corp_name', 'change_on', 'mxmm_shrholdr_nm', 'posesn_stock_co', 'qota_rt', 'change_cause', 'rm', 'stlm_dt', 'bsns_year', 'reprt_code', 'stock_code', 'report_name']

=== 미리보기 ===


,rcept_no,corp_cls,corp_code,corp_name,stock_knd,acqs_mth1,acqs_mth2,acqs_mth3,bsis_qy,change_qy_acqs,change_qy_dsps,change_qy_incnr,trmend_qy,rm,stlm_dt,bsns_year,reprt_code,stock_code,report_name
0,20200330003851,Y,00126380,삼성전자,-,-,-,-,-,-,-,-,-,-,2019-12-31,2019,11011,005930,사업보고서
1,20190814002218,Y,00126380,삼성전자,-,-,-,-,-,-,-,-,-,-,2019-06-30,2019,11012,005930,반기보고서
2,20190515001605,Y,00126380,삼성전자,-,-,-,-,-,-,-,-,-,-,2019-03-31,2019,11013,005930,1분기보고서
3,20191114001273,Y,00126380,삼성전자,-,-,-,-,-,-,-,-,-,-,2019-09-30,2019,11014,005930,3분기보고서
4,20210309000744,Y,00126380,삼성전자,-,-,-,-,-,-,-,-,-,-,2020-12-31,2020,11011,005930,사업보고서
5,20200814001766,Y,00126380,삼성전자,-,-,-,-,-,-,-,-,-,-,2020-06-30,2020,11012,005930,반기보고서
6,20200515001451,Y,00126380,삼성전자,-,-,-,-,-,-,-,-,-,-,2020-03-31,2020,11013,005930,1분기보고서
7,20201116001248,Y,00126380,삼성전자,-,-,-,-,-,-,-,-,-,-,2020-09-30,2020,11014,005930,3분기보고서
8,20220308000798,Y,00126380,삼성전자,-,-,-,-,-,-,-,-,-,-,2021-12-31,2021,11011,005930,사업보고서
9,20210817001416,Y,00126380,삼성전자,-,-,-,-,-,-,-,-,-,-,2021-06-30,2021,11012,005930,반기보고서


,rcept_no,corp_cls,corp_code,corp_name,stock_knd,nm,relate,bsis_posesn_stock_co,bsis_posesn_stock_qota_rt,trmend_posesn_stock_co,trmend_posesn_stock_qota_rt,rm,stlm_dt,bsns_year,reprt_code,stock_code,report_name
0,20200330003851,Y,00126380,삼성전자,보통주,이건희,최대주주 본인,249273200,4.18,249273200,4.18,-,2019-12-31,2019,11011,005930,사업보고서
1,20200330003851,Y,00126380,삼성전자,보통주,삼성생명보험㈜(특별계정),계열회사,19048733,0.32,18286593,0.31,장내매매,2019-12-31,2019,11011,005930,사업보고서
2,20200330003851,Y,00126380,삼성전자,우선주,삼성생명보험㈜(특별계정),계열회사,1268546,0.15,1352563,0.16,장내매매,2019-12-31,2019,11011,005930,사업보고서
3,20200330003851,Y,00126380,삼성전자,보통주,삼성화재해상보험㈜,계열회사,88802052,1.49,88802052,1.49,-,2019-12-31,2019,11011,005930,사업보고서
4,20200330003851,Y,00126380,삼성전자,보통주,이상훈,발행회사 임원,28500,0.00,16000,0.00,장내매매,2019-12-31,2019,11011,005930,사업보고서
5,20200330003851,Y,00126380,삼성전자,보통주,김기남,발행회사 임원,175000,0.00,200000,0.00,장내매매,2019-12-31,2019,11011,005930,사업보고서
6,20200330003851,Y,00126380,삼성전자,보통주,김현석,발행회사 임원,99750,0.00,99750,0.00,-,2019-12-31,2019,11011,005930,사업보고서
7,20200330003851,Y,00126380,삼성전자,보통주,고동진,발행회사 임원,50000,0.00,75000,0.00,장내매매,2019-12-31,2019,11011,005930,사업보고서
8,20200330003851,Y,00126380,삼성전자,보통주,안규리,발행회사 임원,0,0.00,800,0.00,장내매매,2019-12-31,2019,11011,005930,사업보고서
9,20200330003851,Y,00126380,삼성전자,보통주,김한조,발행회사 임원,0,0.00,2175,0.00,장내매매,2019-12-31,2019,11011,005930,사업보고서


,rcept_no,corp_cls,corp_code,corp_name,change_on,mxmm_shrholdr_nm,posesn_stock_co,qota_rt,change_cause,rm,stlm_dt,bsns_year,reprt_code,stock_code,report_name
0,20200330003851,Y,00126380,삼성전자,-,-,-,-,-,-,2019-12-31,2019,11011,005930,사업보고서
1,20190814002218,Y,00126380,삼성전자,-,-,-,-,-,-,2019-06-30,2019,11012,005930,반기보고서
2,20190515001605,Y,00126380,삼성전자,-,-,-,-,-,-,2019-03-31,2019,11013,005930,1분기보고서
3,20191114001273,Y,00126380,삼성전자,-,-,-,-,-,-,2019-09-30,2019,11014,005930,3분기보고서
4,20210309000744,Y,00126380,삼성전자,-,-,-,-,-,-,2020-12-31,2020,11011,005930,사업보고서
5,20200814001766,Y,00126380,삼성전자,-,-,-,-,-,-,2020-06-30,2020,11012,005930,반기보고서
6,20200515001451,Y,00126380,삼성전자,-,-,-,-,-,-,2020-03-31,2020,11013,005930,1분기보고서
7,20201116001248,Y,00126380,삼성전자,-,-,-,-,-,-,2020-09-30,2020,11014,005930,3분기보고서
8,20220308000798,Y,00126380,삼성전자,2021년 04월 29일,삼성생명보험㈜,1263050053,21.16,변동전 최대주주의 피상속,-,2021-12-31,2021,11011,005930,사업보고서
9,20210817001416,Y,00126380,삼성전자,2021년 04월 29일,삼성생명보험㈜,1263050053,21.16,변경전 최대주주의 피상속,-,2021-06-30,2021,11012,005930,반기보고서


In [24]:
import pandas as pd

def to_num(x):
    if pd.isna(x):
        return 0
    x = str(x).replace(",", "").replace("%", "").strip()
    return 0 if x in ["", "-", "nan", "None"] else float(x)

stock_code = "005930"

# 1) 삼성전자 최대주주 현황 최신 기준일
major_s = major_df[major_df["stock_code"] == stock_code].copy()
major_s["stlm_dt"] = pd.to_datetime(major_s["stlm_dt"], errors="coerce")
major_s["trmend_posesn_stock_co_num"] = major_s["trmend_posesn_stock_co"].map(to_num)

latest_major_dt = major_s["stlm_dt"].max()
major_latest = major_s[major_s["stlm_dt"] == latest_major_dt].copy()

# 최대주주 본인
major_owner_only = major_latest.loc[
    major_latest["relate"].astype(str).str.contains("최대주주 본인", na=False),
    "trmend_posesn_stock_co_num"
].sum()

# 특수관계인 포함 전체
major_total_including_related = major_latest["trmend_posesn_stock_co_num"].sum()

# 2) 삼성전자 자기주식 최신 기준일
treasury_s = treasury_df[treasury_df["stock_code"] == stock_code].copy()
treasury_s["stlm_dt"] = pd.to_datetime(treasury_s["stlm_dt"], errors="coerce")
treasury_s["trmend_qy_num"] = treasury_s["trmend_qy"].map(to_num)

latest_treasury_dt = treasury_s["stlm_dt"].max()
treasury_latest = treasury_s[treasury_s["stlm_dt"] == latest_treasury_dt].copy()

# 보통주+우선주 포함 자기주식 총합
treasury_total = treasury_latest["trmend_qy_num"].sum()

# 3) 결과 출력
result = pd.DataFrame([{
    "종목코드": stock_code,
    "회사명": "삼성전자",
    "최대주주기준일": latest_major_dt.date() if pd.notna(latest_major_dt) else None,
    "자기주식기준일": latest_treasury_dt.date() if pd.notna(latest_treasury_dt) else None,
    "최대주주_본인_보유주식수": int(major_owner_only),
    "최대주주_특수관계인포함_총보유주식수": int(major_total_including_related),
    "자기주식수_총합": int(treasury_total),
    "본인보유+자기주식": int(major_owner_only + treasury_total),
    "특수관계인포함총보유+자기주식": int(major_total_including_related + treasury_total),
}])

display(result)

print("=== 최대주주 최신 상세 ===")
display(
    major_latest[[
        "nm", "relate", "stock_knd",
        "trmend_posesn_stock_co", "trmend_posesn_stock_qota_rt", "stlm_dt"
    ]].reset_index(drop=True)
)

print("=== 자기주식 최신 상세 ===")
display(
    treasury_latest[[
        "stock_knd", "trmend_qy", "stlm_dt"
    ]].reset_index(drop=True)
)

,종목코드,회사명,최대주주기준일,자기주식기준일,최대주주_본인_보유주식수,최대주주_특수관계인포함_총보유주식수,자기주식수_총합,본인보유+자기주식,특수관계인포함총보유+자기주식
0,005930,삼성전자,2025-12-31,2025-12-31,508533047,2350381328,316297344,824830391,2666678672


=== 최대주주 최신 상세 ===


,nm,relate,stock_knd,trmend_posesn_stock_co,trmend_posesn_stock_qota_rt,stlm_dt
0,삼성생명보험㈜,최대주주 본인,보통주,503904843,8.51,2025-12-31
1,홍라희,최대주주의 특수관계인,보통주,87978700,1.49,2025-12-31
2,홍라희,최대주주의 특수관계인,우선주,206633,0.03,2025-12-31
3,이재용,최대주주의 특수관계인,보통주,97414196,1.65,2025-12-31
4,이재용,최대주주의 특수관계인,우선주,137757,0.02,2025-12-31
5,이부진,계열회사 임원,보통주,41745681,0.71,2025-12-31
6,이부진,계열회사 임원,우선주,137755,0.02,2025-12-31
7,이서현,계열회사 임원,보통주,45574190,0.77,2025-12-31
8,이서현,계열회사 임원,우선주,137755,0.02,2025-12-31
9,전영현,계열회사 임원,보통주,17000,0.00,2025-12-31


=== 자기주식 최신 상세 ===


,stock_knd,trmend_qy,stlm_dt
0,보통주,91828987,2025-12-31
1,우선주,-,2025-12-31
2,보통주,-,2025-12-31
3,우선주,-,2025-12-31
4,보통주,-,2025-12-31
5,우선주,-,2025-12-31
6,보통주,-,2025-12-31
7,우선주,-,2025-12-31
8,보통주,91828987,2025-12-31
9,우선주,13603461,2025-12-31


In [ ]:
import io
import time
import zipfile
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from pathlib import Path

API_KEY = "여기에_본인_OPENDART_API_KEY"
TXT_PATH = Path(r"./20252코스피.txt")   # 필요시 파일 경로 수정
OUT_PATH = Path(r"./dart_output/전체종목_특수관계인포함총보유_자기주식수.csv")
OUT_PATH.parent.mkdir(exist_ok=True)

BASE_URL = "https://opendart.fss.or.kr/api"
REPORT_CODES = {
    "사업보고서": "11011",
    "반기보고서": "11012",
    "1분기보고서": "11013",
    "3분기보고서": "11014",
}
ENDPOINTS = {
    "corp_code": f"{BASE_URL}/corpCode.xml",
    "treasury_stock": f"{BASE_URL}/tesstkAcqsDspsSttus.json",
    "major_shareholder": f"{BASE_URL}/hyslrSttus.json",
}

def safe_request(url, params=None, timeout=30, max_retries=3, sleep_sec=0.15):
    last_err = None
    for i in range(max_retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            time.sleep(sleep_sec)
            return r
        except Exception as e:
            last_err = e
            time.sleep(1 + i)
    raise last_err

def dart_json(url, params):
    r = safe_request(url, params=params)
    data = r.json()
    status = data.get("status")
    if status not in ("000", "013"):
        raise RuntimeError(f"DART 오류: status={status}, message={data.get('message')}")
    return data

def list_to_df(data):
    rows = data.get("list", [])
    return pd.DataFrame(rows) if rows else pd.DataFrame()

def to_num(x):
    if pd.isna(x):
        return 0.0
    x = str(x).replace(",", "").replace("%", "").strip()
    return 0.0 if x in ["", "-", "nan", "None"] else float(x)

def download_corp_codes(api_key: str) -> pd.DataFrame:
    r = safe_request(ENDPOINTS["corp_code"], params={"crtfc_key": api_key}, timeout=60)
    with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
        xml_name = zf.namelist()[0]
        xml_bytes = zf.read(xml_name)

    root = ET.fromstring(xml_bytes)
    rows = []
    for item in root.findall("list"):
        rows.append({
            "corp_code": item.findtext("corp_code", default="").strip(),
            "corp_name": item.findtext("corp_name", default="").strip(),
            "stock_code": item.findtext("stock_code", default="").strip(),
            "modify_date": item.findtext("modify_date", default="").strip(),
        })
    df = pd.DataFrame(rows)
    df["stock_code"] = df["stock_code"].astype(str).str.zfill(6)
    df = df[df["stock_code"] != ""].copy()
    return df

def read_target_names(txt_path: Path):
    with open(txt_path, "r", encoding="utf-8") as f:
        names = [line.strip() for line in f if line.strip()]
    return names

def build_name_map(corp_map: pd.DataFrame):
    m = corp_map.copy()
    m["corp_name_norm"] = (
        m["corp_name"].astype(str)
        .str.replace("㈜", "", regex=False)
        .str.replace("(주)", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.upper()
    )
    return m

def find_stock_code_by_name(name: str, corp_map_norm: pd.DataFrame):
    name_norm = (
        str(name).strip()
        .replace("㈜", "")
        .replace("(주)", "")
        .replace(" ", "")
        .upper()
    )

    exact = corp_map_norm[corp_map_norm["corp_name_norm"] == name_norm]
    if not exact.empty:
        row = exact.iloc[0]
        return row["stock_code"], row["corp_code"], row["corp_name"]

    contains = corp_map_norm[
        corp_map_norm["corp_name_norm"].str.contains(name_norm, na=False) |
        pd.Series([name_norm]).repeat(len(corp_map_norm)).reset_index(drop=True).combine(
            corp_map_norm["corp_name_norm"], lambda a, b: a in b if isinstance(b, str) else False
        )
    ]
    if not contains.empty:
        row = contains.iloc[0]
        return row["stock_code"], row["corp_code"], row["corp_name"]

    return None, None, None

def get_major_shareholder_status(corp_code: str, bsns_year: int, reprt_code: str) -> pd.DataFrame:
    data = dart_json(ENDPOINTS["major_shareholder"], {
        "crtfc_key": API_KEY,
        "corp_code": corp_code,
        "bsns_year": str(bsns_year),
        "reprt_code": reprt_code,
    })
    df = list_to_df(data)
    if not df.empty:
        df["corp_code"] = corp_code
        df["bsns_year"] = bsns_year
        df["reprt_code"] = reprt_code
    return df

def get_treasury_stock_status(corp_code: str, bsns_year: int, reprt_code: str) -> pd.DataFrame:
    data = dart_json(ENDPOINTS["treasury_stock"], {
        "crtfc_key": API_KEY,
        "corp_code": corp_code,
        "bsns_year": str(bsns_year),
        "reprt_code": reprt_code,
    })
    df = list_to_df(data)
    if not df.empty:
        df["corp_code"] = corp_code
        df["bsns_year"] = bsns_year
        df["reprt_code"] = reprt_code
    return df

def collect_one_company_latest(corp_name: str, stock_code: str, corp_code: str, years=range(2019, 2026)):
    major_list = []
    treasury_list = []

    for year in years:
        for report_name, reprt_code in REPORT_CODES.items():
            try:
                mdf = get_major_shareholder_status(corp_code, year, reprt_code)
                if not mdf.empty:
                    mdf["stock_code"] = stock_code
                    mdf["종목"] = corp_name
                    mdf["report_name"] = report_name
                    major_list.append(mdf)
            except Exception:
                pass

            try:
                tdf = get_treasury_stock_status(corp_code, year, reprt_code)
                if not tdf.empty:
                    tdf["stock_code"] = stock_code
                    tdf["종목"] = corp_name
                    tdf["report_name"] = report_name
                    treasury_list.append(tdf)
            except Exception:
                pass

    major_df = pd.concat(major_list, ignore_index=True) if major_list else pd.DataFrame()
    treasury_df = pd.concat(treasury_list, ignore_index=True) if treasury_list else pd.DataFrame()

    # 최대주주 특수관계인 포함 총보유주식수: 최신 결산기준일 기준 합계
    if not major_df.empty:
        major_df["stlm_dt"] = pd.to_datetime(major_df["stlm_dt"], errors="coerce")
        major_df["trmend_posesn_stock_co_num"] = major_df["trmend_posesn_stock_co"].map(to_num)
        latest_major_dt = major_df["stlm_dt"].max()
        major_total = major_df.loc[
            major_df["stlm_dt"] == latest_major_dt, "trmend_posesn_stock_co_num"
        ].sum()
    else:
        latest_major_dt = pd.NaT
        major_total = 0.0

    # 자기주식수: 최신 결산기준일 기준 합계
    if not treasury_df.empty:
        treasury_df["stlm_dt"] = pd.to_datetime(treasury_df["stlm_dt"], errors="coerce")
        treasury_df["trmend_qy_num"] = treasury_df["trmend_qy"].map(to_num)
        latest_treasury_dt = treasury_df["stlm_dt"].max()
        treasury_total = treasury_df.loc[
            treasury_df["stlm_dt"] == latest_treasury_dt, "trmend_qy_num"
        ].sum()
    else:
        latest_treasury_dt = pd.NaT
        treasury_total = 0.0

    return {
        "종목": corp_name,
        "종목코드": stock_code,
        "corp_code": corp_code,
        "최대주주최신기준일": latest_major_dt.date() if pd.notna(latest_major_dt) else None,
        "자기주식최신기준일": latest_treasury_dt.date() if pd.notna(latest_treasury_dt) else None,
        "특수관계인포함총보유주식수": int(major_total),
        "자기주식수": int(treasury_total),
        "특수관계인포함총보유+자기주식수": int(major_total + treasury_total),
    }

# 1) 종목명 리스트 읽기
target_names = read_target_names(TXT_PATH)

# 2) corp_code 매핑 테이블 준비
corp_map = download_corp_codes(API_KEY)
corp_map_norm = build_name_map(corp_map)

# 3) 종목명 -> stock_code / corp_code 매핑
mapped = []
unmatched = []

for name in target_names:
    stock_code, corp_code, matched_name = find_stock_code_by_name(name, corp_map_norm)
    if stock_code is None:
        unmatched.append(name)
    else:
        mapped.append({
            "입력종목명": name,
            "종목": matched_name,
            "종목코드": stock_code,
            "corp_code": corp_code
        })

mapped_df = pd.DataFrame(mapped).drop_duplicates(subset=["종목코드"]).reset_index(drop=True)
print("매핑 성공:", len(mapped_df))
print("매핑 실패:", len(unmatched))
if unmatched:
    print("매핑 실패 종목:", unmatched[:30])

# 4) 전체 종목 수집
results = []
for i, row in mapped_df.iterrows():
    print(f"[{i+1}/{len(mapped_df)}] {row['종목']} ({row['종목코드']})")
    try:
        out = collect_one_company_latest(
            corp_name=row["종목"],
            stock_code=row["종목코드"],
            corp_code=row["corp_code"],
            years=range(2019, 2026)
        )
        results.append(out)
    except Exception as e:
        print("  -> 실패:", e)
        results.append({
            "종목": row["종목"],
            "종목코드": row["종목코드"],
            "corp_code": row["corp_code"],
            "최대주주최신기준일": None,
            "자기주식최신기준일": None,
            "특수관계인포함총보유주식수": 0,
            "자기주식수": 0,
            "특수관계인포함총보유+자기주식수": 0,
        })

result_df = pd.DataFrame(results)

# 5) 요청 컬럼만 저장
final_df = result_df[["종목", "특수관계인포함총보유+자기주식수"]].copy()
final_df = final_df.sort_values(["특수관계인포함총보유+자기주식수", "종목"], ascending=[False, True]).reset_index(drop=True)

final_df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print("\n저장 완료:", OUT_PATH.resolve())
display(final_df.head(20))

매핑 성공: 178
매핑 실패: 5
매핑 실패 종목: ['KT&G', 'LS ELECTRIC', 'SK바이오팜', 'HD현대인프라코어', 'TKG휴켐스']
[1/178] 삼성전자 (005930)
[2/178] SK하이닉스 (000660)
[3/178] LG에너지솔루션 (373220)
[4/178] 삼성바이오로직스 (207940)
[5/178] 사단법인 현대차미소금융재단 (000000)
[6/178] HD현대중공업 (329180)
[7/178] 두산에너빌리티 (034020)
[8/178] KB금융 (105560)
[9/178] 한화에어로스페이스 (012450)
[10/178] 삼성물산 (000830)
[11/178] SK스퀘어 (402340)
[12/178] NAVER (035420)
[13/178] 신한지주 (055550)
[14/178] 한화오션 (042660)
[15/178] 현대모비스 (012330)
[16/178] 고려아연 (010130)
[17/178] HD한국조선해양 (009540)
[18/178] 삼성생명 (032830)
[19/178] HD현대일렉트릭 (267260)
[20/178] LG화학 (051910)
[21/178] POSCO홀딩스 (005490)
[22/178] 하나금융지주 (086790)
[23/178] 삼성SDI (006400)
[24/178] 삼성중공업 (010140)
[25/178] 포스코퓨처엠 (003670)
[26/178] 우리금융지주 (053000)
[27/178] 삼성전기 (009150)
[28/178] 현대로템 (064350)
[29/178] HMM (011200)
[30/178] 메리츠금융지주 (138040)
[31/178] SK이노베이션 (096770)
[32/178] SK (003600)
[33/178] 삼성에피스홀딩스 (0126Z0)
[34/178] 효성중공업 (298040)
[35/178] 기업은행 (024110)
[36/178] HD현대 (267250)
[37/178] LG전자 (066570)
[38/178] 

,종목,특수관계인포함총보유+자기주식수
0,LS,20670512000
1,삼성전자,2666678672
2,한화생명,1134046150
3,한온시스템,743700576
4,HMM,668266854
5,팬오션,587168510
6,삼성중공업,445030973
7,NH투자증권,443511637
8,대우건설,421892418
9,두산에너빌리티,392976290
